In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
create widget text storageName default "adlaiza082026";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
%python
spark.sql(f"""
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore@{storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev'
""")

In [0]:
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

###Tablas Bronze

In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.bronze.goodreads_books (
  title string, series string, author string, rating double,
  description string, language string, isbn string, genres string,
  characters string, bookFormat string, edition string, pages integer,
  publisher string, publishDate string, firstPublishDate string,
  awards string, numRatings long, ratingsByStars string, likedPercent double,
  setting string, bbeScore long, bbeVotes long, price double, ingestion_date timestamp
)
USING DELTA
LOCATION 'abfss://bronze@{storageName}.dfs.core.windows.net/goodreads_books'
""")

In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.bronze.amazon_bestsellers (
  name string, author string, user_rating double, reviews integer,
  price integer, year integer, genre string, ingestion_date timestamp
)
USING DELTA
LOCATION 'abfss://bronze@{storageName}.dfs.core.windows.net/amazon_bestsellers'
""")

In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.bronze.country_reading (
  country string, books_read_annually double, hours_reading_per_year integer,
  ingestion_date timestamp
)
USING DELTA
LOCATION 'abfss://bronze@{storageName}.dfs.core.windows.net/country_reading'
""")

###Tablas Silver

In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.silver.books_catalog_transformed (
  title string, series string, author string, rating double, language string,
  isbn string, primary_genre string, genres_clean string, pages integer,
  publisher string, publish_date date, num_ratings long, liked_percent double,
  price double, ingestion_date timestamp
)
USING DELTA
LOCATION 'abfss://silver@{storageName}.dfs.core.windows.net/books_catalog_transformed'
""")

In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.silver.bestsellers_transformed (
  name string, author string, user_rating double, reviews integer,
  price integer, year integer, genre string, title_norm string,
  author_norm string, ingestion_date timestamp
)
USING DELTA
LOCATION 'abfss://silver@{storageName}.dfs.core.windows.net/bestsellers_transformed'
""")

In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.silver.catalog_vs_bestsellers (
  bestseller_name string, bestseller_author string, bestseller_genre string,
  year integer, in_catalog boolean, catalog_title string,
  catalog_primary_genre string, catalog_rating double, ingestion_date timestamp
)
USING DELTA
LOCATION 'abfss://silver@{storageName}.dfs.core.windows.net/catalog_vs_bestsellers'
""")

In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.silver.country_reading_transformed (
  country string, books_read_annually double, hours_reading_per_year integer,
  reading_rank integer, ingestion_date timestamp
)
USING DELTA
LOCATION 'abfss://silver@{storageName}.dfs.core.windows.net/country_reading_transformed'
""")

###Tablas Golden

In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.golden.top_authors (
  author string, total_bestseller_books long, avg_user_rating double, years_active integer
)
USING DELTA
LOCATION 'abfss://golden@{storageName}.dfs.core.windows.net/top_authors'
""")

In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.golden.top_books (
  name string, author string, years_as_bestseller integer, avg_user_rating double, genre string
)
USING DELTA
LOCATION 'abfss://golden@{storageName}.dfs.core.windows.net/top_books'
""")

In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.golden.genre_distribution (
  genre string, book_count long, avg_rating double
)
USING DELTA
LOCATION 'abfss://golden@{storageName}.dfs.core.windows.net/genre_distribution'
""")

In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.golden.top_books_by_genre (
  genre string, rank integer, title string, author string, rating double
)
USING DELTA
LOCATION 'abfss://golden@{storageName}.dfs.core.windows.net/top_books_by_genre'
""")

In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.golden.country_reading (
  country string, reading_index double, rank integer
)
USING DELTA
LOCATION 'abfss://golden@{storageName}.dfs.core.windows.net/country_reading'
""")